# Practical Session: Low-Frequency Meteorological Data Preprocessing

**CRYOEXTREMES Course — Practical Workflows**  
*Penguin Island Field Station Observation Processing*

--- 

## 🎯 Notebook Overview
This notebook handles the ingesting, parsing, time-synchronization, and quality control (QC) of low-frequency meteorological data recorded at the Penguin Island flux tower site.

### Key Operations:
1. **Data Ingestion**: Parses multi-file log outputs (`Pinguim_b*` radiation/met files and `Pinguim_w*` wind files).
2. **Timestamp Construction**: Converts Year, Day of Year (DOY), and HHMM timestamps into continuous UTC `datetime` objects.
3. **Quality Control & Resampling**: Aggregates raw observations to 1-hour temporal resolution while filtering out hourly windows with low sampling rates (`min_count < 50`).
4. **Dataset Alignment & Export**: Merges radiative, thermodynamic, and wind datasets into a unified pandas DataFrame and exports the clean dataset for Surface Energy Balance (SEB) modeling.

--- 
## 1. Setup & Ingestion Functions

We define a modular ingestion function `process_tower_file_group` to handle both radiation/thermodynamic parameters and wind parameters without repeating code.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

def process_tower_file_group(base_path, file_pattern, columns, min_count=50):
    """
    Read, parse, time-index, and resample tower logger files.

    Parameters
    ----------
    base_path : str
        Directory containing raw data files.
    file_pattern : str
        Pattern to match logger files (e.g., 'Pinguim_b*', 'Pinguim_w*').
    columns : list of str
        List of variable column names corresponding to the logger output.
    min_count : int, default=50
        Minimum number of valid records required per hour for QC filtering.

    Returns
    -------
    df_hourly : pandas.DataFrame
        Hourly resampled DataFrame indexed by UTC datetime with QC applied.
    """
    # 1. Locate matching files
    folder_path = os.path.join(base_path)
    files = sorted(glob.glob(os.path.join(folder_path, file_pattern)))
    
    if not files:
        raise FileNotFoundError(f"No files found matching pattern '{file_pattern}' in '{base_path}'")

    # 2. Read and concatenate data files
    df_list = []
    for file in files:
        df_tmp = pd.read_csv(file, header=None)
        df_tmp.columns = columns
        df_list.append(df_tmp)

    df = pd.concat(df_list, ignore_index=True)

    # 3. Construct Datetime Index (Year + DOY + Hour/Minute)
    df['hh'] = df['hour'] // 100
    df['mm'] = df['hour'] % 100

    df['datetime'] = (
        pd.to_datetime(df['year'], format='%Y') +
        pd.to_timedelta(df['doy'] - 1, unit='D') +
        pd.to_timedelta(df['hh'], unit='h') +
        pd.to_timedelta(df['mm'], unit='m')
    )

    df = df.drop(columns=['hh', 'mm'])

    # 4. Set datetime index & remove duplicate timestamps
    df = df.set_index('datetime').sort_index()
    df = df[~df.index.duplicated(keep='first')]

    # 5. Resample to 1-hour resolution with QC record count masking
    df_mean = df.resample('1h').mean()
    df_count = df.resample('1h').count()

    df_hourly = df_mean.copy()
    df_hourly[df_count < min_count] = np.nan

    return df_hourly


--- 
## 2. Ingest Radiation, Thermodynamics, and Wind Data

We specify column definitions for the main data files (`Pinguim_b*`) and wind files (`Pinguim_w*`) and process both groups.

In [2]:
# Base path setup (Compatible with relative repo structure)
base_path = os.path.join("data", "AP", "low_freq")

# Column definitions
b_vars = ['id', 'year', 'doy', 'hour', 'sec', 'SUp', 'SDn', 'LUp', 'LDn', 'CNR4TC', 
          'CNR4TK', 'RsNet', 'RlNet', 'Albedo', 'RN', 'LUpCo', 'LDnCo', 'tair', 'rh', 
          'press', 'SHF']

w_vars = ['id', 'year', 'doy', 'hour', 'sec', 'ws', 'wd']

# Process both file groups
df_hourly_met = process_tower_file_group(base_path, file_pattern='Pinguim_b*', columns=b_vars, min_count=50)
df_hourly_ws  = process_tower_file_group(base_path, file_pattern='Pinguim_w*', columns=w_vars, min_count=50)

print(f"Processed Radiation/Met records: {len(df_hourly_met)} hours")
print(f"Processed Wind records:          {len(df_hourly_ws)} hours")


Processed Radiation/Met records: 193 hours
Processed Wind records:          193 hours


--- 
## 3. Merge Datasets & Validate Synchronization

We join the meteorological/radiation variables with the wind speed/direction variables using an inner index join on `datetime`.

In [3]:
df_met_ws = pd.merge(
    df_hourly_met, 
    df_hourly_ws[['ws', 'wd']], 
    left_index=True, 
    right_index=True, 
    how="inner"
)

print("Merged DataFrame Preview:")
display(df_met_ws[['tair', 'rh', 'press', 'ws', 'wd', 'SUp', 'SDn', 'LUpCo', 'LDnCo', 'SHF']].head())


Merged DataFrame Preview:


,tair,rh,press,ws,wd,SUp,SDn,LUpCo,LDnCo,SHF
datetime,,,,,,,,,,
2024-12-05 00:00:00,-0.765254,76.260000,1009.343220,0.0,0.705288,-1.621271,-0.645508,278.950847,-308.547458,-8.388136
2024-12-05 01:00:00,-1.202200,77.911333,1009.343383,0.0,0.667983,-2.890317,-0.873267,249.340000,-305.000000,-7.664483
2024-12-05 02:00:00,-1.549267,78.135333,1009.448533,0.0,0.632567,-1.856967,-1.455250,239.385000,-302.746667,-6.404567
2024-12-05 03:00:00,-1.609117,77.461833,1009.647250,0.0,0.601150,7.741833,-2.950833,254.713333,-303.516667,-5.897800
2024-12-05 04:00:00,-1.122300,76.896667,1009.670067,0.0,0.648150,73.923833,-20.231583,252.225000,-308.655000,-5.436317


--- 
## 4. Export Clean Hourly Dataset

Select essential Surface Energy Balance (SEB) input variables and write to `met_data_1h.csv`.

In [4]:
# Define export directory & path
output_dir = os.path.join("data", "pross")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "met_data_1h.csv")

# Select target SEB variables
export_vars = ['SUp', 'SDn', 'LUpCo', 'LDnCo', 'tair', 'rh', 'press', 'ws', 'wd', 'SHF']

# Export to CSV
df_met_ws[export_vars].to_csv(output_file)
print(f"Successfully exported {len(df_met_ws)} hourly records to '{output_file}'")


Successfully exported 193 hourly records to 'data/pross/met_data_1h.csv'
